In [ ]:
test

In [ ]:
Reviewed and validated S1 mapping for Clinical Form Answers.form_ans_src_answer.

Definition requires CTV3Code from silver_sone_srcode to be concatenated with form_ans_bridge_form_ans_conformed_answer from silver_rdm_form_answer_bridging.

Validation completed in PROD using SELECT-only queries:
- Confirmed sc.ctv3_code matches br.form_ans_bridge_src_id.
- Confirmed CONCAT('SONE', sc.id_organisation_source) matches br.form_ans_bridge_src_sys_inst_src_id.
- Confirmed COALESCE(selc.question_heading, dermc.question_heading) matches br.form_ans_bridge_form_ques_src_name.

Join condition includes question context to avoid incorrect bridge matches / row multiplication where the same CTV3 code exists under different source questions.

Implementation:
- Added LEFT JOIN to silver_rdm_form_answer_bridging using code + source instance + question name.
- Added form_ans_src_answer as:
  CONCAT_WS('_', sc.ctv3_code, br.form_ans_bridge_form_ans_conformed_answer)

Used underscore separator to stay consistent with existing source ID concatenation pattern.

In [ ]:
SELECT
    sc.ctv3_code,
    br.form_ans_bridge_form_ans_conformed_answer,
    CONCAT(sc.ctv3_code, br.form_ans_bridge_form_ans_conformed_answer) AS form_ans_src_answer
FROM silver_sone_srcode sc
LEFT JOIN silver_rdm_derm_read_codes dermc
    ON sc.ctv3_code = dermc.code
LEFT JOIN silver_rdm_sel_read_codes selc
    ON sc.ctv3_code = selc.code
LEFT JOIN silver_rdm_form_answer_bridging br
    ON TRIM(LOWER(sc.ctv3_code)) = TRIM(LOWER(br.form_ans_bridge_src_id))
   AND TRIM(LOWER(CONCAT('SONE', sc.id_organisation_source))) = TRIM(LOWER(br.form_ans_bridge_src_sys_inst_src_id))
   AND TRIM(LOWER(COALESCE(selc.question_heading, dermc.question_heading))) = TRIM(LOWER(br.form_ans_bridge_form_ques_src_name))
WHERE br.form_ans_bridge_form_ans_conformed_answer IS NOT NULL
LIMIT 20;